# Vidu Q2 Reference-to-Video API 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台调用 **Vidu Q2 (Reference-to-Video)** 模型 API。

## 模型简介

Vidu Q2 Reference-to-Video 是一款基于参考图片生成视频的模型，支持两种调用模式：

### 模式一：非主体调用
- 使用 `reference_image_urls` 传入 1-7 张参考图片
- 模型会参考图片内容生成视频，但不锁定特定主体
- 适合场景参考、风格参考等

### 模式二：主体调用
- 使用 `subjects` 传入主体信息（含 `id`、`images`、`voice_id`）
- 通过 `@主体id` 在 prompt 中引用主体
- 支持 **视频直出**（`bgm: true`）和 **音视频直出**（`audio: true`，prompt 中包含台词）

**API 端点**：`fal-ai/vidu/q2/reference-to-video`

## 1. 环境准备

首先安装 `fal-client` 依赖包，并设置 API Key 和七牛 AIToken 队列服务地址。

In [ ]:
# 安装 fal-client
%pip install fal-client -q

In [ ]:
import os

# 设置七牛 AIToken 队列服务地址
os.environ["FAL_QUEUE_RUN_HOST"] = "api.qnaigc.com/queue"

# 设置 FAL_KEY 环境变量
# 你也可以在终端中通过 export FAL_KEY="xxx" 来设置
os.environ["FAL_KEY"] = os.environ.get("QINIU_API_KEY") or (_ for _ in ()).throw(ValueError("环境变量 QINIU_API_KEY 未设置"))

In [ ]:
import fal_client
from IPython.display import Video, display

In [ ]:
import time

def run_fal_task(endpoint, arguments, poll_interval=5):
    """提交任务并轮询等待结果返回"""
    # 步骤 1：提交请求
    handler = fal_client.submit(endpoint, arguments=arguments)
    request_id = handler.request_id
    print(f"请求已提交，request_id: {request_id}")

    # 步骤 2：轮询任务状态
    while True:
        status = fal_client.status(endpoint, request_id, with_logs=True)
        print(f"当前状态: {type(status).__name__}")

        if isinstance(status, fal_client.Completed):
            print("任务完成!")
            break

        if hasattr(status, "logs") and status.logs:
            for log in status.logs:
                print(log["message"])

        time.sleep(poll_interval)

    # 步骤 3：获取结果
    return fal_client.result(endpoint, request_id)

## 2. 模式一：非主体调用

使用 `reference_image_urls` 传入参考图片，模型参考图片内容生成视频。适合场景参考、风格参考等用途。

- 支持 1-7 张参考图片
- 必填参数：`prompt`、`aspect_ratio`、`bgm`、`reference_image_urls`

In [ ]:
# 非主体调用：使用参考图片生成视频
result_ref = run_fal_task(
    "fal-ai/vidu/q2/reference-to-video",
    arguments={
        "prompt": "一只小猫在花园中好奇地探索，阳光透过树叶洒在地上，温馨的画面",
        "aspect_ratio": "16:9",
        "bgm": True,
        "reference_image_urls": [
            "https://v3.fal.media/files/zebra/sgsdKvPigPhJ1S7Hl5bWc_first_frame_q1.png",
        ],
    },
)

# 打印返回结果
print(result_ref)

In [ ]:
# 展示生成的视频
video_url = result_ref["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

### 非主体调用参数说明

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `prompt` | string | *必填* | 文本提示词，最多 2000 字符 |
| `aspect_ratio` | string | *必填* | 宽高比：`16:9`、`9:16`、`3:4`、`4:3`、`1:1` |
| `bgm` | boolean | *必填* | 是否添加背景音乐 |
| `reference_image_urls` | array | *必填* | 参考图片 URL 列表，1-7 张 |
| `seed` | integer | 随机 | 随机种子 |
| `duration` | integer | `5` | 视频时长（秒） |
| `resolution` | string | `"1080p"` | 分辨率 |
| `movement_amplitude` | string | `"auto"` | 运动幅度：`auto`、`small`、`medium`、`large` |

## 3. 模式二：主体调用（视频直出）

使用 `subjects` 传入主体图片信息，通过 `@主体id` 在 prompt 中引用主体。

视频直出模式：设置 `bgm: true` 添加背景音乐，不含台词对话。

- 支持 1-7 个主体，每个主体最多 3 张图片
- 通过 `@主体id` 在 prompt 中引用对应主体

In [ ]:
# 主体调用 — 视频直出（不含台词，含背景音乐）
result_subject = run_fal_task(
    "fal-ai/vidu/q2/reference-to-video",
    arguments={
        "prompt": "The @devil is looking at the @apple on the @beach and walking around @beach.",
        "subjects": [
            {
                "id": "devil",
                "images": [
                    "https://v3.fal.media/files/zebra/sgsdKvPigPhJ1S7Hl5bWc_first_frame_q1.png",
                ],
                "voice_id": "",
            },
            {
                "id": "apple",
                "images": [
                    "https://v3.fal.media/files/kangaroo/CASBu_OmOnZ8IafirarFL_last_frame_q1.png",
                ],
                "voice_id": "",
            },
            {
                "id": "beach",
                "images": [
                    "https://v3.fal.media/files/zebra/sgsdKvPigPhJ1S7Hl5bWc_first_frame_q1.png",
                ],
                "voice_id": "",
            },
        ],
        "aspect_ratio": "16:9",
        "bgm": True,
        "duration": 5,
        "resolution": "1080p",
        "movement_amplitude": "auto",
    },
)

# 打印返回结果
print(result_subject)

In [ ]:
# 展示主体调用生成的视频
video_url = result_subject["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 4. 模式三：主体调用（音视频直出）

在主体调用基础上启用 `audio: true`，并在 prompt 中添加台词，即可让主体"说话"，直接生成带对白的音视频。

- 设置 `audio: true` 启用音视频直出
- 在 prompt 中通过台词描述让主体发声
- 可通过 `voice_id` 为不同主体设置不同音色

In [ ]:
# 主体调用 — 音视频直出（含台词对话）
result_audio = run_fal_task(
    "fal-ai/vidu/q2/reference-to-video",
    arguments={
        "prompt": "@host 站在舞台中央，面带微笑地说：'大家好，欢迎来到今天的节目！'，然后向观众挥手致意。",
        "subjects": [
            {
                "id": "host",
                "images": [
                    "https://v3.fal.media/files/zebra/sgsdKvPigPhJ1S7Hl5bWc_first_frame_q1.png",
                ],
                "voice_id": "",  # 留空时系统自动推荐音色
            },
        ],
        "audio": True,  # 启用音视频直出
        "aspect_ratio": "16:9",
        "bgm": False,
        "duration": 5,
        "resolution": "1080p",
        "movement_amplitude": "auto",
    },
)

# 打印返回结果
print(result_audio)

In [ ]:
# 展示音视频直出生成的视频
video_url = result_audio["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

### 主体调用参数说明

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `prompt` | string | *必填* | 文本提示词，使用 `@主体id` 引用主体，最多 2000 字符 |
| `subjects` | array | *必填* | 主体信息列表，1-7 个主体 |
| `subjects[].id` | string | *必填* | 主体 ID，用于在 prompt 中通过 `@id` 引用 |
| `subjects[].images` | array | *必填* | 主体图片 URL 列表，每个主体最多 3 张 |
| `subjects[].voice_id` | string | *必填* | 音色 ID，留空时系统自动推荐 |
| `audio` | boolean | `false` | 是否启用音视频直出（含台词对话） |
| `aspect_ratio` | string | *必填* | 宽高比：`16:9`、`9:16`、`3:4`、`4:3`、`1:1` |
| `bgm` | boolean | *必填* | 是否添加背景音乐 |
| `seed` | integer | 随机 | 随机种子 |
| `duration` | integer | `5` | 视频时长（秒） |
| `resolution` | string | `"1080p"` | 分辨率 |
| `movement_amplitude` | string | `"auto"` | 运动幅度：`auto`、`small`、`medium`、`large` |

## 5. 队列模式 — 手动分步调用

上面的 `run_fal_task` 封装了完整的队列流程。如果你需要更精细的控制（例如在提交后做其他事情，稍后再取结果），可以手动分步调用。

In [ ]:
# 步骤 1：提交请求
handler = fal_client.submit(
    "fal-ai/vidu/q2/reference-to-video",
    arguments={
        "prompt": "一只可爱的猫咪在窗台上晒太阳，慵懒地打了个哈欠",
        "aspect_ratio": "16:9",
        "bgm": True,
        "reference_image_urls": [
            "https://v3.fal.media/files/zebra/sgsdKvPigPhJ1S7Hl5bWc_first_frame_q1.png",
        ],
    },
)

# 获取请求 ID，用于后续查询
request_id = handler.request_id
print(f"请求已提交，request_id: {request_id}")

In [ ]:
# 步骤 2：查询任务状态
import time

while True:
    status = fal_client.status(
        "fal-ai/vidu/q2/reference-to-video",
        request_id,
        with_logs=True,
    )
    print(f"当前状态: {type(status).__name__}")

    if isinstance(status, fal_client.Completed):
        print("任务完成!")
        break

    # 每 5 秒查询一次
    time.sleep(5)

In [ ]:
# 步骤 3：获取结果
result_async = fal_client.result(
    "fal-ai/vidu/q2/reference-to-video",
    request_id,
)

# 展示生成的视频
video_url = result_async["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 6. Schema 参考

### Input Schema（非主体调用）

```json
{
  "prompt": "string (必填, 最多 2000 字符)",
  "aspect_ratio": "16:9 | 9:16 | 3:4 | 4:3 | 1:1 (必填)",
  "bgm": "boolean (必填)",
  "reference_image_urls": ["string (图片 URL, 1-7 张)"],
  "seed": "integer (可选, 默认随机)",
  "duration": "integer (可选, 默认 5)",
  "resolution": "1080p (可选, 默认 1080p)",
  "movement_amplitude": "auto | small | medium | large (可选, 默认 auto)"
}
```

### Input Schema（主体调用）

```json
{
  "prompt": "string (必填, 使用 @主体id 引用主体)",
  "subjects": [
    {
      "id": "string (主体 ID)",
      "images": ["string (图片 URL, 每个主体最多 3 张)"],
      "voice_id": "string (音色 ID, 留空自动推荐)"
    }
  ],
  "audio": "boolean (可选, 是否音视频直出)",
  "aspect_ratio": "16:9 | 9:16 | 3:4 | 4:3 | 1:1 (必填)",
  "bgm": "boolean (必填)",
  "seed": "integer (可选, 默认随机)",
  "duration": "integer (可选, 默认 5)",
  "resolution": "1080p (可选, 默认 1080p)",
  "movement_amplitude": "auto | small | medium | large (可选, 默认 auto)"
}
```

### Output Schema (队列提交响应)

```json
{
  "status": "IN_QUEUE",
  "request_id": "string (任务唯一 ID)",
  "response_url": "string (查询任务结果的 URL)",
  "status_url": "string (查询任务状态的 URL)",
  "cancel_url": "string (取消任务的 URL, 暂未支持)"
}
```

### 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [fal-client Python 文档](https://docs.fal.ai/reference/client-libraries/python/fal_client)
- [七牛 API 调用示例文档](https://apidocs.qnaigc.com)